### Establish connection

In [9]:
import sys; sys.path.append("..")
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv
from datetime import datetime, timezone
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from utils import ping_openai

from openai_client import openai_client
from mongo_client import mongo_client, articles_collection, test_articles_collection

try:
    from dateutil import parser as date_parser
except ImportError:
    date_parser = None

load_dotenv()

try:
    mongo_client.admin.command("ping")
    print("✅ Connected to MongoDB Atlas!")
except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    raise

# establish openai client 
ping_openai() 

✅ Connected to MongoDB Atlas!
✅ Successfully connected to OpenAI API!


In [3]:
# Return unique instances of a variable

unique_validations = articles_collection.distinct("validation")
print(unique_validations)

# return count of entries that have a validation value

count_with_validation = articles_collection.count_documents({"validation": {"$exists": True}})
print(count_with_validation) 

# return count of entries for which validation is True

count_with_validation = articles_collection.count_documents({"validation": False})
print(count_with_validation) 

[1747217798, 1747226078, 1747228805, 1747229679, 1747231393, 1747231725, 1747232234, 1747234713, 1747235063, 1747235304, 1747236099, 1747236296, 1747664759, 1749483121, 1750089285, 1750162669, 1750165867, 1750175030, 1750230919, 1750230981, 1750231089, 1750231137, 1750231251, 1750231478, 1750337048, 1750337349, 1750337823, 1750341403, 1750346232, 1750409348, 1750411179, 1750411352, 1750687343, 1750704301, 1750704981, 1750706676, 1750707305, 1750707730, 1750708119, 1750708978, 1750755109, 1750755310, 1750755420, 1750755793, 1750756317, 1750756863, 1750757238, 1750758436, 1750759171, 1750759257, 1750759537, 1750759726, 1750780457, 1750842889, 1750844184, 1750848408, 1750848649, 1750848908, 1750849032, 1750849368, 1750849884, 1750850927, 1750851257, 1750851621, 1750855054, 1750856312, 1750862715, 1750863118, 1750863499, 1750863762, 1750864141, 1750864295, 1750866015, 1750866049, 1750867472, 1750867731, 1750869637, 1750870011, 1750870288, 1750870542, 1750871066, 1750871961, 1750872045, 175

In [4]:
results = list(articles_collection.find(
    {"validated": {"$exists": True}},
    {"title": 1, "meta.date": 1, "_id": 0}
))

titles = [doc["title"] for doc in results]
print(titles)

dates = [doc["meta"]["date"] for doc in results if "meta" in doc and "date" in doc["meta"]]
print(dates)

['Mahle integrates DC charger to climatic wind tunnel', 'Honda could soon make EVs and batteries in Canada', 'ZF upgrades US plant for electrified drives', 'LS e-Mobility to manufacture EV components in Mexico', 'Cupra kicks off production of the all-electric Tavascan in China', 'Ford opens electric drive testing lab in the UK', 'BMW launches operation of battery test centre in Wackersdorf', 'Cornish Lithium opens demonstration plant in Cornwall', 'Funding for battery recycling plant from ABTC', 'Moment Energy to construct EV battery reuse Gigafactory in Texas', 'Forsee Power inaugurates factory in Ohio', 'Ford pauses construction of Canadian cathode material plant again', 'BTR plans to produce anode materials in Morocco', 'Funding for battery recycling plant from ABTC', 'Elgin Energy Finds Greek Partner for 76MW UK PV', 'Sterling and Wilson Wins $141 Million Deal for 500 MW Solar Project in India']
['2023-02-09T00:00:00.000Z', '2024-01-08T00:00:00.000Z', '2024-02-07T00:00:00.000Z', '2

In [5]:
cutoff_date = datetime(2025, 1, 1)

results = list(articles_collection.find(
    {
        "meta.date": {"$gt": cutoff_date}
    },
    {
        "title": 1,
        "meta.date": 1,
        "_id": 0
    }
))

In [6]:
results

[{'title': 'Mexico wants to build affordable electric cars',
  'meta': {'date': datetime.datetime(2025, 1, 8, 0, 0)}},
 {'title': 'Sony-Honda alliance prices electric saloon Afeela 1 at $90,000',
  'meta': {'date': datetime.datetime(2025, 1, 7, 0, 0)}},
 {'title': 'Honda presents two concept cars at CES 2025',
  'meta': {'date': datetime.datetime(2025, 1, 8, 0, 0)}},
 {'title': 'VW: Is Zwickau’s future still uncertain?',
  'meta': {'date': datetime.datetime(2025, 1, 10, 0, 0)}},
 {'title': 'Hyundai close to starting solid-state battery pilot production',
  'meta': {'date': datetime.datetime(2025, 1, 2, 0, 0)}},
 {'title': 'Rolls-Royce invests in electric production',
  'meta': {'date': datetime.datetime(2025, 1, 8, 0, 0)}},
 {'title': 'USA changes EV tax credit list: Hyundai is in, VW is out',
  'meta': {'date': datetime.datetime(2025, 1, 6, 0, 0)}},
 {'title': 'BYD gears up to introduce the Atto 2 in Europe',
  'meta': {'date': datetime.datetime(2025, 1, 10, 0, 0)}},
 {'title': 'Phoen

### General search

In [15]:
offset = 0
limit = 1

articles_to_process = list(
    test_articles_collection.find(
        {"factory_geonames_enriched_at": {"$exists": True}},
        {"_id": 1, "paragraphs": 1, "validation": 1, "meta": 1, "title": 1,
         "nodes":1, "relationships":1}
    )
    .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
    .skip(offset)    # Skip first `offset` articles
    .limit(limit)    # Limit the number of articles
)

for article in articles_to_process:
    print(article["title"])

    val = article.get("validation")

    if val is True:
        print("⏭️  Skipping – article is validated")
        continue
    elif isinstance(val, (int, float)):           # timestamp
        processed_on = datetime.fromtimestamp(val, tz=timezone.utc) \
                                 .strftime("%Y‑%m‑%d %H:%M UTC")
        print(f"⏭️  Skipping – article was validated on {processed_on}")
        continue

Terrapower Expands Cooperation With Japan on Fast Reactors


In [16]:
articles_to_process

[{'_id': ObjectId('67fbf79f94b7f855cfea7169'),
  'title': 'Terrapower Expands Cooperation With Japan on Fast Reactors',
  'paragraphs': [{'p1': "In the strategic roadmap for fast reactor development adopted by Japan's Cabinet in December 2018, a policy was defined to assess the efficacy of various types of fast reactors to be developed following a technological competition among private-sector corporations. The roadmap was subsequently revised by the Cabinet decision in December 2022, at which time two decisions were taken: firstly, to select a sodium-cooled fast reactor as the target of the conceptual design of the demonstration reactor, set to get under way in fiscal 2024; and secondly, to select a manufacturer to serve as the core company in charge of the fast reactor's design and requisite R&D which would proceed with technology development in accordance with the goals and policy directions established by the government.",
    'p2': 'In July this year, MHI was selected by the Japan

In [10]:
query = {
    "factory_geonames_enriched_at": {"$exists": False},  # skip already-processed
    "nodes": {
        "$elemMatch": {
            "type": "factory",
            "location.city": {"$exists": True, "$nin": [None, "", "null", "nan"]},
            "location.country": {"$exists": True, "$nin": [None, "", "null", "nan"]},
            "$or": [
                {"location.factory_city_adm1_name": {"$exists": False}},
                {"location.factory_city_adm1_name": None}
            ]
        }
    }
}

entries = list(test_articles_collection.find(query))
print(f"Found {len(entries)} article(s) with unenriched factory location(s)")

Found 402 article(s) with unenriched factory location(s)


### Validation metrics

In [4]:
validated_count = articles_collection.count_documents({"validation": True})
print(f"Number of validated articles: {validated_count}")

Number of validated articles: 168


### Publication date metrics

In [5]:
# Check meta.date consistency
def classify_date(value):
    if isinstance(value, datetime):
        return "bson"
    if isinstance(value, str):
        if date_parser:
            try:
                date_parser.parse(value)
                return "string"
            except (ValueError, OverflowError):
                return "invalid"
    return "invalid"

In [6]:
counts = {"bson": 0, "string": 0, "invalid": 0, "missing": 0}
invalid_ids = []

cursor = articles_collection.find({}, {"_id": 1, "meta.date": 1}, batch_size=10_000)
total = 0

for doc in cursor:
    total += 1
    meta = doc.get("meta", {})
    if "date" not in meta:
        counts["missing"] += 1
        invalid_ids.append(doc["_id"])
        continue
    category = classify_date(meta["date"])
    counts[category] += 1
    if category == "invalid":
        invalid_ids.append(doc["_id"])

In [7]:
print("\n--- meta.date consistency report ---")
print(f"Total documents: {total:,}")
for key in ["bson", "string", "invalid", "missing"]:
    n = counts[key]
    pct = (n / total * 100) if total else 0
    print(f"{key.capitalize():>8}: {n:>7,}  ({pct:5.1f}%)")


--- meta.date consistency report ---
Total documents: 39,578
    Bson:  39,318  ( 99.3%)
  String:      57  (  0.1%)
 Invalid:     203  (  0.5%)
 Missing:       0  (  0.0%)


In [8]:
# return all date entries
cursor = articles_collection.find(
    {"meta.date": {"$type": "date"}},
    {"_id": 0, "meta.date": 1}
)
dates = [doc["meta"]["date"].date() for doc in tqdm(cursor)]
df = pd.DataFrame(dates, columns=["date"])

daily_counts = df.value_counts().reset_index(name="count").sort_values("date")
daily_counts.columns = ["date", "count"]
daily_counts["date"] = pd.to_datetime(daily_counts["date"])  # ensure correct dtype
daily_counts["year"] = daily_counts["date"].dt.year

output_dir = "descriptive/daily_counts"

# save one plot per year
years = sorted(daily_counts["year"].unique())
for year in years:
    fig, ax = plt.subplots(figsize=(12, 4))
    year_df = daily_counts[daily_counts["year"] == year]
    ax.plot(year_df["date"], year_df["count"], marker="", linewidth=1)
    ax.set_title(f"Daily Article Count – {year}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Articles per Day")
    ax.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()

    fig_path = os.path.join(output_dir, f"daily_count_{year}.png")
    plt.savefig(fig_path)
    plt.close(fig)

print(f"✅ Saved {len(years)} plots to {output_dir}")

39318it [00:00, 109476.04it/s]


✅ Saved 19 plots to descriptive/daily_counts


### Duplicate entries

In [9]:
# return all URLs
cursor = articles_collection.find(
    {"meta.url": {"$exists": True}},  # ensure url field exists
    {"_id": 1, "meta.url": 1}
)

# build dataframe
data = [{"_id": doc["_id"], "url": doc["meta"]["url"]} for doc in tqdm(cursor)]
df_urls = pd.DataFrame(data)

# count duplicate URLs
url_counts = df_urls["url"].value_counts()
duplicates = url_counts[url_counts > 1]

# display result
print(f"🔍 Total articles: {len(df_urls)}")
print(f"❗ Duplicate URLs found: {len(duplicates)}")
print(duplicates.head(40))  # top 10 most duplicated URLs

38564it [00:01, 37105.42it/s]

🔍 Total articles: 38564
❗ Duplicate URLs found: 1401
url
https://www.world-energy.org/tag/150.html                                                                                                8
https://www.world-energy.org/tag/244.html                                                                                                7
https://www.world-energy.org/tag/176.html                                                                                                6
https://www.pv-magazine.com/author/sergiomatalucci/                                                                                      5
https://www.pv-magazine.com/author/uma-gupta/                                                                                            5
https://www.world-energy.org/article/39139.html                                                                                          5
https://www.world-energy.org/article/32160.html                                                              

In [10]:
# Step 1: Query for all titles
cursor = articles_collection.find(
    {"title": {"$exists": True}},  # only documents with title
    {"_id": 1, "title": 1}
)

# Step 2: Build dataframe
data = [{"_id": doc["_id"], "title": doc["title"]} for doc in tqdm(cursor)]
df_titles = pd.DataFrame(data)

# Step 3: Count duplicate titles
title_counts = df_titles["title"].value_counts()
duplicate_titles = title_counts[title_counts > 1]

# Step 4: Display result
print(f"🔍 Total articles: {len(df_titles)}")
print(f"❗ Duplicate titles found: {len(duplicate_titles)}")
print(duplicate_titles.head(40))

38564it [00:00, 52900.53it/s]

🔍 Total articles: 38564
❗ Duplicate titles found: 1656
title
No Title Found                                                                                        1590
Utility Scale PV                                                                                       601
Modules & Upstream Manufacturing                                                                       412
Hydrogen                                                                                                72
                                                                                                        48
Energy Storage                                                                                          35
Balance of Systems                                                                                      26
The pv magazine weekly news digest                                                                      11
Solar Power Plant                                                                  

In [17]:
# Step 1: Filter only titles that are duplicates
duplicate_title_counts = title_counts[title_counts > 1]

# Step 2: Count how many titles appear 2 times, 3 times, etc.
dup_frequency_distribution = duplicate_title_counts.value_counts().sort_index(ascending=False)

# Step 3: Display the distribution
print("🧮 Distribution of duplicate title frequencies:")
print(dup_frequency_distribution)

🧮 Distribution of duplicate title frequencies:
count
1592       1
601        1
412        1
73         1
35         1
26         1
11         1
10         1
8          3
7          3
6          9
5         15
4         80
3        257
2       1302
Name: count, dtype: int64


### by article source category

In [9]:
pipeline = [
    {"$group": {"_id": "$meta.category", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}         # optional, highest count first
]

for doc in articles_collection.aggregate(pipeline):
    print(f"{doc['_id']}: {doc['count']}")

world_energy: 9808
pvmagazine: 9433
pvtech: 7709
electrive: 4868
renewsBiz: 4360
powertechnology: 1437
offshorewind: 665
energy_voice_hydrogen: 284


In [14]:
from pprint import pprint 
pipeline = [
    # 1. Collapse to the two fields we care about
    {
        "$project": {
            "category": {"$ifNull": ["$meta.category", "⟨no-category⟩"]},
            "run_id":   "$llm_processed.run_id"
        }
    },

    # 2. Group by BOTH category and run_id
    {
        "$group": {
            "_id": {"category": "$category", "run_id": "$run_id"},
            "count": {"$sum": 1}
        }
    },

    # 3. Regroup by category to bundle the run_id breakdown
    {
        "$group": {
            "_id": "$_id.category",
            "total":   {"$sum": "$count"},
            "by_run":  {
                "$push": {         # keep each run-id + its count
                    "run_id": "$_id.run_id",
                    "count" : "$count"
                }
            }
        }
    },

    { "$sort": { "total": -1 } }               # optional pretty ordering
]

results = list(articles_collection.aggregate(pipeline))
pprint(results)

[{'_id': 'world_energy',
  'by_run': [{'count': 8813}, {'count': 995, 'run_id': 'v0'}],
  'total': 9808},
 {'_id': 'pvmagazine', 'by_run': [{'count': 9433}], 'total': 9433},
 {'_id': 'pvtech', 'by_run': [{'count': 7709}], 'total': 7709},
 {'_id': 'electrive',
  'by_run': [{'count': 445, 'run_id': 'v0'}, {'count': 4423}],
  'total': 4868},
 {'_id': 'renewsBiz', 'by_run': [{'count': 4360}], 'total': 4360},
 {'_id': 'powertechnology', 'by_run': [{'count': 1437}], 'total': 1437},
 {'_id': 'offshorewind', 'by_run': [{'count': 665}], 'total': 665},
 {'_id': 'energy_voice_hydrogen', 'by_run': [{'count': 284}], 'total': 284}]
